In [4]:
import pandas as pd 
car=pd.read_csv("quikr_car.csv")
car.head()

,name,company,year,Price,kms_driven,fuel_type
0,Hyundai Santro Xing XO eRLX Euro III,Hyundai,2007,"80,000","45,000 kms",Petrol
1,Mahindra Jeep CL550 MDI,Mahindra,2006,"4,25,000",40 kms,Diesel
2,Maruti Suzuki Alto 800 Vxi,Maruti,2018,Ask For Price,"22,000 kms",Petrol
3,Hyundai Grand i10 Magna 1.2 Kappa VTVT,Hyundai,2014,"3,25,000","28,000 kms",Petrol
4,Ford EcoSport Titanium 1.5L TDCi,Ford,2014,"5,75,000","36,000 kms",Diesel


In [5]:
car.shape

(892, 6)

In [41]:
car.info()

<class 'pandas.core.frame.DataFrame'>
Index: 817 entries, 0 to 889
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   name        817 non-null    object
 1   company     817 non-null    object
 2   year        817 non-null    int64 
 3   Price       817 non-null    int32 
 4   kms_driven  817 non-null    int32 
 5   fuel_type   816 non-null    object
dtypes: int32(2), int64(1), object(3)
memory usage: 38.3+ KB


## problems
##### year has many non year values 
##### year object to int 
##### price has some string values 
##### price object to int
##### kms_driven has kms with integers
##### kms_driven object to int 
##### kms_driven has nan values 
##### fuel_type has nan values 
##### keep first 3 words of name

# Cleaning

In [7]:
backup=car.copy()

In [8]:
car=car[car['year'].str.isnumeric()]

In [9]:
car['year']=car['year'].astype(int)

In [18]:
car=car[car['Price']!='Ask For Price']

In [22]:
car['Price']=car['Price'].str.replace(',','')

In [23]:
car['Price']=car['Price'].astype('int32')

In [47]:
car['kms_driven']=car['kms_driven'].str.split(' ').str.get(0).str.replace(',','')
car=car[car['kms_driven'].str.isnumeric()]
car['kms_driven']=car['kms_driven'].astype('int32')

In [52]:
car=car[~car['fuel_type'].isnull()]#exclude null value rows 

In [57]:
car['name']=car['name'].str.split(' ').str.slice(0,3).str.join(' ')

In [61]:
#reset index 
car=car.reset_index(drop=True)#drop drops previous indexes

In [62]:
car.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 816 entries, 0 to 815
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   name        816 non-null    object
 1   company     816 non-null    object
 2   year        816 non-null    int64 
 3   Price       816 non-null    int32 
 4   kms_driven  816 non-null    int32 
 5   fuel_type   816 non-null    object
dtypes: int32(2), int64(1), object(3)
memory usage: 32.0+ KB


In [63]:
car.describe()

,year,Price,kms_driven
count,816.000000,8.160000e+02,816.000000
mean,2012.444853,4.117176e+05,46275.531863
std,4.002992,4.751844e+05,34297.428044
min,1995.000000,3.000000e+04,0.000000
25%,2010.000000,1.750000e+05,27000.000000
50%,2013.000000,2.999990e+05,41000.000000
75%,2015.000000,4.912500e+05,56818.500000
max,2019.000000,8.500003e+06,400000.000000


In [71]:
#75% cars are below 600000
car=car[car['Price']<6e6].reset_index(drop=True)

In [72]:
car.to_csv('Cleaned_car.csv')

## Model

In [119]:
x=car.drop(columns='Price')
y=car['Price']

In [120]:
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2)

In [121]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from sklearn.preprocessing import OneHotEncoder

In [122]:
#onehot encoder converts text into numerical format
ohe=OneHotEncoder()
ohe.fit(x[['name','company','fuel_type']])
ohe.categories_

[array(['Audi A3 Cabriolet', 'Audi A4 1.8', 'Audi A4 2.0', 'Audi A6 2.0',
        'Audi A8', 'Audi Q3 2.0', 'Audi Q5 2.0', 'Audi Q7', 'BMW 3 Series',
        'BMW 5 Series', 'BMW 7 Series', 'BMW X1', 'BMW X1 sDrive20d',
        'BMW X1 xDrive20d', 'Chevrolet Beat', 'Chevrolet Beat Diesel',
        'Chevrolet Beat LS', 'Chevrolet Beat LT', 'Chevrolet Beat PS',
        'Chevrolet Cruze LTZ', 'Chevrolet Enjoy', 'Chevrolet Enjoy 1.4',
        'Chevrolet Sail 1.2', 'Chevrolet Sail UVA', 'Chevrolet Spark',
        'Chevrolet Spark 1.0', 'Chevrolet Spark LS', 'Chevrolet Spark LT',
        'Chevrolet Tavera LS', 'Chevrolet Tavera Neo', 'Datsun GO T',
        'Datsun Go Plus', 'Datsun Redi GO', 'Fiat Linea Emotion',
        'Fiat Petra ELX', 'Fiat Punto Emotion', 'Force Motors Force',
        'Force Motors One', 'Ford EcoSport', 'Ford EcoSport Ambiente',
        'Ford EcoSport Titanium', 'Ford EcoSport Trend',
        'Ford Endeavor 4x4', 'Ford Fiesta', 'Ford Fiesta SXi', 'Ford Figo',
        '

In [123]:
#ColumnTransformer: Applies different transformations to different columns.
#Pipeline: Connects preprocessing and model training into one process.
#simple imputer removes nan values 
from sklearn.compose import make_column_transformer
from sklearn.pipeline import make_pipeline 
column_trans=make_column_transformer((OneHotEncoder(categories=ohe.categories_),['name','company','fuel_type']),remainder='passthrough')
lr=LinearRegression()
pipe=make_pipeline(column_trans,lr)
pipe.fit(x_train,y_train)

C:\Users\RGUKT\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\compose\_column_transformer.py:1667: FutureWarning: 
The format of the columns of the 'remainder' transformer in ColumnTransformer.transformers_ will change in version 1.7 to match the format of the other transformers.
At the moment the remainder columns are stored as indices (of type int). With the same ColumnTransformer configuration, in the future they will be stored as column names (of type str).
To use the new behavior now and suppress this warning, use ColumnTransformer(force_int_remainder_cols=False).

  warnings.warn(


Pipeline(steps=[('columntransformer',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('onehotencoder',
                                                  OneHotEncoder(categories=[array(['Audi A3 Cabriolet', 'Audi A4 1.8', 'Audi A4 2.0', 'Audi A6 2.0',
       'Audi A8', 'Audi Q3 2.0', 'Audi Q5 2.0', 'Audi Q7', 'BMW 3 Series',
       'BMW 5 Series', 'BMW 7 Series', 'BMW X1', 'BMW X1 sDrive20d',
       'BMW X1 xDrive20d', 'Chevrolet Beat', 'Chevrolet Beat...
                                                                            array(['Audi', 'BMW', 'Chevrolet', 'Datsun', 'Fiat', 'Force', 'Ford',
       'Hindustan', 'Honda', 'Hyundai', 'Jaguar', 'Jeep', 'Land',
       'Mahindra', 'Maruti', 'Mercedes', 'Mini', 'Mitsubishi', 'Nissan',
       'Renault', 'Skoda', 'Tata', 'Toyota', 'Volkswagen', 'Volvo'],
      dtype=object),
                                                                            array(['Diesel', 'LPG', 'Petrol'], dtype=object)]),
                                                  ['name', 'company',
                                                   'fuel_type'])])),
                ('linearregression', LinearRegression())])

In [124]:
y_pred=pipe.predict(x_test)
y_pred

array([ 306665.52966301,  155478.40785317,  309748.72752945,
         93829.23052006,  679538.55230197,  450816.5703579 ,
        400904.26423438,  341186.97355646,   11377.13862828,
       -132555.44996504,  456007.18410252,   70420.66416489,
        261176.72127171,  446956.97729675,  280062.01391673,
        612640.27403729,  447073.86619334,   19328.66546483,
        282051.16960105,  247715.68168866,  335565.57943691,
        372204.15164032,   49072.61643413,  426900.17958755,
        123111.15236302,  518771.27217247,  175394.12127994,
        177805.8246988 ,  365209.99347178,  343671.8779626 ,
        779251.16524621,  435002.30337383,  604669.4010388 ,
        646647.14800899,  280062.01391673,  292433.04685058,
        359191.32850078,  738286.15119889,  193887.80809145,
        407910.29923514,  101747.32639669,  642476.09461057,
        278858.98011234,  143735.2168715 ,  611177.17372019,
        243903.4906741 ,  342432.91366615,  520279.06856817,
        285989.76292555,

In [105]:
r2_score(y_test,y_pred)

0.7408241433512663

In [129]:
scores=[]
for i in range(1000):
    x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=i)
    lr=LinearRegression()
    pipe=make_pipeline(column_trans,lr)
    pipe.fit(x_train,y_train)
    y_pred=pipe.predict(x_test)
    scores.append(r2_score(y_test,y_pred))

In [138]:
import numpy as np 
np.argmax(scores)#returns position of best score 


np.int64(433)

In [137]:
 scores[np.argmax(scores)]

0.8457059012561223

In [140]:
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=np.argmax(scores))
lr=LinearRegression()
pipe=make_pipeline(column_trans,lr)
pipe.fit(x_train,y_train)
y_pred=pipe.predict(x_test)
r2_score(y_test,y_pred)

0.8457059012561223

In [141]:
import pickle

In [143]:
pickle.dump(pipe, open('LinearRegressionModel.pkl','wb'))

In [145]:
pipe.predict(pd.DataFrame([['Maruti Suzuki Swift','Maruti',2019,100,'Petrol']],columns=['name','company','year','kms_driven','fuel_type']))

array([458894.10960853])